## DermaVision AI — Gold Model (Multimodal Fusion)
**Team:** Group 7 | Kishore, Ellie Lansdown, Ryan Bennett, Grace Callahan, Stephanie Furst
**Course:** MSBC 5190 Modern AI for Business | Spring 2026

---

### What Gold adds over Silver 2.0

Silver 2.0 is a pure image classifier — three vision models, weighted ensemble, TTA, calibration.
Gold introduces **multimodal fusion**: the model sees patient metadata alongside the image,
and for MILK10k lesions it also sees *paired* clinical + dermoscopic images of the same lesion.

| Stream | Input | Model |
|--------|-------|-------|
| Dermoscopic image | 224×224 dermoscope photo | Silver 2.0 ensemble (frozen) |
| Clinical image | 224×224 smartphone/clinical photo | EfficientNetB0 (new, MILK10k clinical stream) |
| Patient metadata | Age, sex, Fitzpatrick type, lesion location, diameter | Small MLP encoder |

These streams are fused via **late fusion** (concatenate + MLP classifier) as the baseline,
with a commented-out cross-attention alternative for the stretch goal.

```
[Dermoscopic img] → Silver2.0 embeddings (Eff+Swin+CLIP) ──┐
[Clinical img]    → ClinicalStream (EfficientNetB0)        ──┤→ Concat(1088d) → FusionMLP → 9 classes
[Metadata]        → MetadataEncoder (MLP, 30→64d)          ──┘
```

### Prerequisites

1. Google Drive with Silver 2.0 checkpoints:
   - `MyDrive/DERMAVISION/checkpoints/effnet_silver2_best.keras`
   - `MyDrive/DERMAVISION/checkpoints/swinv2_silver2_best.pt`
   - `MyDrive/DERMAVISION/checkpoints/biomedclip_silver2_best.pt`

2. If checkpoints are missing, falls back to ImageNet pre-trained weights (slower to converge).

> **Note for Kishore:** Each section has `# TODO` comments marking where implementation is needed.
> Sections 1–7 (data, metadata, DataLoaders, model loading) are fully written.
> Sections 8–11 (clinical stream, fusion, training) need you to run and debug.
> Section 12 (Streamlit app) has a working UI scaffold — just connect the model inference.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installs — run once, restart runtime if prompted
!pip install -q timm open_clip_torch pytorch-grad-cam transformers
!pip install -q kagglehub scikit-learn einops

# Kaggle credentials (choose one option):
# Option A: upload kaggle.json manually, then:
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# Option B: load from Drive:
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import os, warnings, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import open_clip
import tensorflow as tf
from tensorflow.keras import mixed_precision

from sklearn.model_selection import train_test_split
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix, recall_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import kagglehub

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
tf.random.set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

mixed_precision.set_global_policy('mixed_float16')

## 2. Configuration & Label Map

Same 9-class canonical map as Silver 2.0. Gold does not add new classes —
the goal is to improve *performance and fairness* on the existing 9, not expand scope.
(Expanding to 15 conditions requires sourcing and labelling additional data — post-Gold goal.)


In [ ]:
LABEL_NAMES = ['mel', 'nv', 'bcc', 'scc', 'akiec', 'bkl', 'df', 'vasc', 'unk']
LABEL2IDX   = {l: i for i, l in enumerate(LABEL_NAMES)}
N_CLASSES   = len(LABEL_NAMES)
HIGH_RISK   = {'mel', 'bcc', 'scc'}
MOD_RISK    = {'akiec'}

CKPT_DIR  = '/content/drive/MyDrive/DERMAVISION/checkpoints'
GOLD_DIR  = '/content/drive/MyDrive/DERMAVISION/gold_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32

# Silver 2.0 Nelder-Mead optimised weights (for reference)
SILVER_WEIGHTS = {'eff': 0.20, 'swin': 0.50, 'clip': 0.30}

print('Config loaded. Classes:', LABEL_NAMES)

## 3. Dataset Loading

Gold uses the same three Silver 2.0 training datasets **plus** MILK10k **clinical** images.

The key structural difference from Silver 2.0: every row in the combined DataFrame now has
two image paths — `path_derm` (dermoscopic) and `path_clin` (clinical). For ISIC 2019
and PAD-UFES-20 rows that have no clinical image, `path_clin` is set to the same path as
`path_derm`. This is a simple and defensible fallback — the clinical stream will learn that
"same image twice" means "no extra information available."

> **Alternative Kishore could explore:** A learnable "missing stream" token/embedding
> instead of duplicating the dermoscopic image.


In [ ]:
print('Downloading datasets...')
isic_path  = kagglehub.dataset_download('andrewmvd/isic-2019')
pad_path   = kagglehub.dataset_download('mahdavi1202/skin-cancer')
milk_path  = kagglehub.dataset_download('nguyenphucduyloc/milk10k-isic-challenge-2025')
print(f'ISIC 2019 : {isic_path}')
print(f'PAD-UFES  : {pad_path}')
print(f'MILK10k   : {milk_path}')

In [ ]:
# ── ISIC 2019 ──────────────────────────────────────────────────────────────────
ISIC_LABEL_MAP = {
    'MEL':'mel','NV':'nv','BCC':'bcc','AK':'akiec',
    'BKL':'bkl','DF':'df','VASC':'vasc','SCC':'scc','UNK':'unk'
}
isic_gt = None
for fname in ['ISIC_2019_Training_GroundTruth.csv', 'train.csv']:
    p = Path(isic_path) / fname
    if p.exists(): isic_gt = pd.read_csv(p); break
assert isic_gt is not None, 'ISIC ground truth CSV not found'

isic_gt['label'] = isic_gt[list(ISIC_LABEL_MAP)].idxmax(axis=1).map(ISIC_LABEL_MAP)

img_dir_isic = None
for d in ['ISIC_2019_Training_Input', 'train', 'images']:
    p = Path(isic_path) / d
    if p.is_dir(): img_dir_isic = p; break

isic_df = pd.DataFrame({
    'image'      : isic_gt['image'],
    'label'      : isic_gt['label'],
    'path_derm'  : isic_gt['image'].apply(lambda x: str(img_dir_isic / f'{x}.jpg') if img_dir_isic else ''),
    'path_clin'  : isic_gt['image'].apply(lambda x: str(img_dir_isic / f'{x}.jpg') if img_dir_isic else ''),
    'source'     : 'isic_2019',
    'fitzpatrick': np.nan, 'age': np.nan, 'sex': np.nan,
    'location'   : np.nan, 'diameter': np.nan,
})
print(f'ISIC 2019: {len(isic_df):,} images')

In [ ]:
# ── PAD-UFES-20 ────────────────────────────────────────────────────────────────
PAD_LABEL_MAP = {'MEL':'mel','NEV':'nv','BCC':'bcc','SCC':'scc','ACK':'akiec','SEK':'bkl'}

pad_meta = None
for fname in ['metadata.csv', 'pad-ufes-20_parsed_folders.csv']:
    p = Path(pad_path) / fname
    if p.exists(): pad_meta = pd.read_csv(p); break

pad_meta.columns = pad_meta.columns.str.strip().str.lower()
for col in ['fitzpatrick', 'skin_tone', 'fitz', 'fitzpatrick_scale']:
    if col in pad_meta.columns:
        pad_meta = pad_meta.rename(columns={col: 'fitzpatrick'}); break

pad_img_dir = None
for d in ['images', 'imgs', 'data', '']:
    p = Path(pad_path) / d
    if p.is_dir() and any(p.glob('*.png')): pad_img_dir = p; break
if pad_img_dir is None: pad_img_dir = Path(pad_path)

pad_meta['label'] = pad_meta['diagnostic'].str.upper().map(PAD_LABEL_MAP).fillna('unk')
img_id_col_pad    = 'img_id' if 'img_id' in pad_meta.columns else pad_meta.columns[0]

pad_df = pd.DataFrame({
    'image'      : pad_meta[img_id_col_pad],
    'label'      : pad_meta['label'],
    'path_derm'  : pad_meta[img_id_col_pad].apply(lambda x: str(pad_img_dir / x)),
    'path_clin'  : pad_meta[img_id_col_pad].apply(lambda x: str(pad_img_dir / x)),
    'source'     : 'pad_ufes_20',
    'fitzpatrick': pad_meta.get('fitzpatrick', pd.Series(np.nan, index=pad_meta.index)),
    'age'        : pad_meta.get('age',      pd.Series(np.nan, index=pad_meta.index)),
    'sex'        : pad_meta.get('sex',      pd.Series(np.nan, index=pad_meta.index)),
    'location'   : pad_meta.get('region',   pd.Series(np.nan, index=pad_meta.index)),
    'diameter'   : pad_meta.get('diameter_1', pd.Series(np.nan, index=pad_meta.index)),
})
print(f'PAD-UFES-20: {len(pad_df):,} images | '
      f'{pad_df["fitzpatrick"].notna().sum()} Fitzpatrick labels')

In [ ]:
# ── MILK10k — PAIRED loading (key Gold addition) ───────────────────────────────
#
# MILK10k provides 5,240 lesions each with TWO images:
#   path_derm → dermoscopic image  (used in Silver 2.0)
#   path_clin → clinical/smartphone image  (NEW in Gold)
#
# TODO for Kishore: Print the folder structure and confirm path patterns match
# your Kaggle download. The code below covers the most common layouts.

milk_root = Path(milk_path)
print('MILK10k top-level:', [p.name for p in milk_root.iterdir()][:15])

milk_meta = None
for fname in ['metadata.csv', 'train.csv', 'ISIC_2025_TRAIN_metadata.csv']:
    for search_root in [milk_root] + list(milk_root.iterdir()):
        if not search_root.is_dir(): continue
        p = search_root / fname
        if p.exists():
            milk_meta = pd.read_csv(p)
            print(f'Loaded metadata: {p}')
            break
    if milk_meta is not None: break

assert milk_meta is not None, 'MILK10k metadata CSV not found — check folder structure'
milk_meta.columns = milk_meta.columns.str.strip().str.lower()
print(f'Columns: {list(milk_meta.columns)[:15]}')
print(f'Rows: {len(milk_meta):,}')
milk_meta.head(3)

In [ ]:
# ── MILK10k: find image directories and build paired DataFrame ─────────────────
MILK_LABEL_MAP = {
    'mel':'mel','melanoma':'mel','nv':'nv','nevus':'nv','melanocytic nevi':'nv',
    'bcc':'bcc','basal cell carcinoma':'bcc','scc':'scc','squamous cell carcinoma':'scc',
    'akiec':'akiec','actinic keratosis':'akiec','bkl':'bkl',
    'benign keratosis':'bkl','seborrheic keratosis':'bkl',
    'df':'df','dermatofibroma':'df','vasc':'vasc','vascular lesion':'vasc',
}

label_col = None
for c in ['label', 'target', 'diagnosis', 'dx']:
    if c in milk_meta.columns: label_col = c; break

if label_col:
    milk_meta['label'] = milk_meta[label_col].astype(str).str.lower().map(MILK_LABEL_MAP).fillna('unk')
else:
    milk_meta['label'] = 'unk'
    print('WARNING: no label column found — all set to unk')

# Find separate derm / clinical directories
derm_dir = clin_dir = None
for sub in milk_root.iterdir():
    if not sub.is_dir(): continue
    name = sub.name.lower()
    if any(k in name for k in ['derm', 'dermoscop']): derm_dir = sub
    if any(k in name for k in ['clin', 'photo', 'clinical', 'smartphone']): clin_dir = sub

if derm_dir is None: derm_dir = milk_root
print(f'Dermoscopic dir : {derm_dir}')
print(f'Clinical dir    : {clin_dir if clin_dir else "Not found — will duplicate derm path"}')

img_id_col = 'isic_id' if 'isic_id' in milk_meta.columns else milk_meta.columns[0]

milk_df = pd.DataFrame({
    'image'      : milk_meta[img_id_col],
    'label'      : milk_meta['label'],
    'path_derm'  : milk_meta[img_id_col].apply(lambda x: str(derm_dir / f'{x}.jpg')),
    'path_clin'  : milk_meta[img_id_col].apply(
        lambda x: str(clin_dir / f'{x}.jpg') if clin_dir else str(derm_dir / f'{x}.jpg')),
    'source'     : 'milk10k',
    'fitzpatrick': np.nan, 'age': np.nan, 'sex': np.nan,
    'location'   : np.nan, 'diameter': np.nan,
})
print(f'MILK10k: {len(milk_df):,} paired lesions')

In [ ]:
# ── Combine all sources ─────────────────────────────────────────────────────────
metadata = pd.concat([isic_df, pad_df, milk_df], ignore_index=True)

isic_ids = set(isic_df['image']); pad_ids = set(pad_df['image']); milk_ids = set(milk_df['image'])
print(f'ISIC ∩ PAD  overlap: {len(isic_ids & pad_ids)}')
print(f'ISIC ∩ MILK overlap: {len(isic_ids & milk_ids)}')
print(f'PAD  ∩ MILK overlap: {len(pad_ids  & milk_ids)}')

metadata['label_idx'] = metadata['label'].map(LABEL2IDX)
metadata = metadata.dropna(subset=['label_idx'])
metadata['label_idx'] = metadata['label_idx'].astype(int)

print(f'\nTotal combined: {len(metadata):,} images')
print(metadata['source'].value_counts())
print(metadata['label'].value_counts())

## 4. Train / Val / Test Split (70/15/15)

In [ ]:
train_meta, temp_meta = train_test_split(
    metadata, test_size=0.30, random_state=SEED, stratify=metadata['label'])
val_meta, test_meta = train_test_split(
    temp_meta, test_size=0.50, random_state=SEED, stratify=temp_meta['label'])

print(f'Train: {len(train_meta):,} | Val: {len(val_meta):,} | Test: {len(test_meta):,}')

train_meta.to_csv(f'{GOLD_DIR}/gold_train_meta.csv', index=False)
val_meta.to_csv(f'{GOLD_DIR}/gold_val_meta.csv',   index=False)
test_meta.to_csv(f'{GOLD_DIR}/gold_test_meta.csv',  index=False)
print('Splits saved to Drive.')

## 5. Metadata Feature Encoding

PAD-UFES-20 provides rich per-patient metadata; ISIC 2019 and MILK10k do not.

**Strategy: zero-imputation + missingness indicator.**
For each feature that is missing, we set the value to 0 and add a binary indicator bit
(e.g., `age_missing = 1`). This lets the metadata MLP learn that "no metadata available"
is itself a meaningful signal — it's correlated with dataset source, which correlates with
image type (dermoscopic vs smartphone) and skin tone distribution.

**Total: 30 features**
- Age: 2 (value + missing flag)
- Sex: 3 (male, female, unknown — one-hot)
- Fitzpatrick type: 7 (types I–VI + missing — one-hot)
- Lesion location: 16 (14 locations + unknown + missing — one-hot)
- Diameter: 2 (value + missing flag)


In [ ]:
LOCATION_CATS = [
    'back','lower extremity','trunk','upper extremity','face','chest',
    'abdomen','foot','hand','neck','scalp','ear','genital','acral'
]

def encode_metadata(df):
    rows = []
    for _, row in df.iterrows():
        feats = []
        # Age (normalised to [0,1])
        age = row.get('age', np.nan)
        feats += ([0.0, 1.0] if pd.isna(age) else [min(float(age)/100.0, 1.0), 0.0])
        # Sex (one-hot)
        sex = str(row.get('sex', '')).lower()
        feats += [
            1.0 if 'male' in sex and 'fe' not in sex else 0.0,
            1.0 if 'female' in sex else 0.0,
            1.0 if sex in ('', 'nan', 'unknown', 'none') else 0.0,
        ]
        # Fitzpatrick (one-hot 1–6 + missing)
        fitz = row.get('fitzpatrick', np.nan)
        fv = [0.0]*7
        if pd.isna(fitz):
            fv[6] = 1.0
        else:
            idx = int(float(fitz)) - 1
            fv[idx] = 1.0 if 0 <= idx <= 5 else 0.0
            if not (0 <= idx <= 5): fv[6] = 1.0
        feats += fv
        # Location (one-hot + unknown/missing)
        loc = str(row.get('location', '')).lower().strip()
        lv = [0.0]*(len(LOCATION_CATS)+2)
        if loc and loc not in ('', 'nan', 'none'):
            matched = False
            for i, cat in enumerate(LOCATION_CATS):
                if cat in loc: lv[i] = 1.0; matched = True; break
            if not matched: lv[-2] = 1.0   # unknown category
        else:
            lv[-1] = 1.0   # missing
        feats += lv
        # Diameter (normalised, max ~50mm)
        diam = row.get('diameter', np.nan)
        feats += ([0.0, 1.0] if pd.isna(diam) else [min(float(diam)/50.0, 1.0), 0.0])
        rows.append(feats)
    return np.array(rows, dtype=np.float32)

META_DIM = 2 + 3 + 7 + (len(LOCATION_CATS)+2) + 2
print(f'Metadata feature dimension: {META_DIM}')

train_meta_feats = encode_metadata(train_meta)
val_meta_feats   = encode_metadata(val_meta)
test_meta_feats  = encode_metadata(test_meta)
print(f'Train metadata shape: {train_meta_feats.shape}')

## 6. GoldDataset — Paired Image + Metadata

Each `__getitem__` returns 4 tensors:
1. `img_derm` — dermoscopic image (3, 224, 224)
2. `img_clin` — clinical image (3, 224, 224) [same as derm if no paired clinical available]
3. `meta` — metadata feature vector (META_DIM,)
4. `label` — integer class label


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class GoldDataset(Dataset):
    def __init__(self, df, meta_feats, transform=None):
        self.df        = df.reset_index(drop=True)
        self.meta      = meta_feats
        self.transform = transform or val_transform

    def __len__(self): return len(self.df)

    def _load(self, path):
        try:   return self.transform(Image.open(path).convert('RGB'))
        except: return self.transform(Image.new('RGB', (IMG_SIZE, IMG_SIZE), (127,127,127)))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return (self._load(row['path_derm']),
                self._load(row['path_clin']),
                torch.tensor(self.meta[idx], dtype=torch.float32),
                int(row['label_idx']))

train_ds = GoldDataset(train_meta, train_meta_feats, train_transform)
val_ds   = GoldDataset(val_meta,   val_meta_feats,   val_transform)
test_ds  = GoldDataset(test_meta,  test_meta_feats,  val_transform)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 7. Load Silver 2.0 Image Backbones (Feature Extractors)

We load the three Silver 2.0 trained models and use them as **frozen feature extractors**.
Instead of taking the final 9-class softmax, we tap the **penultimate layer (256-dim)**
for richer representations that the fusion layer can work with.

Keeping Silver 2.0 backbones frozen in Gold training has two advantages:
1. No catastrophic forgetting of dermoscopic features learned across 38k images
2. Much faster training — only the fusion layer + clinical stream are updated


In [ ]:
# ── Focal loss (required to load EfficientNetB0 .keras checkpoint) ─────────────
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, class_weight=None, **kw):
        super().__init__(**kw)
        self.gamma = gamma
        self.cw = tf.constant(class_weight, dtype=tf.float32) if class_weight is not None else None
    def call(self, y_true, y_pred):
        y_true   = tf.cast(y_true, tf.int32)
        y_onehot = tf.one_hot(y_true, N_CLASSES)
        ce = -tf.math.log(tf.clip_by_value(y_pred, 1e-7, 1.0))
        pt = tf.reduce_sum(y_onehot * y_pred, axis=-1, keepdims=True)
        fl = ((1-pt)**self.gamma) * ce
        if self.cw is not None:
            w = tf.reduce_sum(y_onehot * self.cw, axis=-1, keepdims=True)
            fl = fl * w
        return tf.reduce_mean(tf.reduce_sum(fl, axis=-1))
    def get_config(self): return {'gamma': self.gamma}

EFF_CKPT = f'{CKPT_DIR}/effnet_silver2_best.keras'
if os.path.exists(EFF_CKPT):
    eff_full = tf.keras.models.load_model(EFF_CKPT, custom_objects={'FocalLoss': FocalLoss})
    print('Loaded EfficientNetB0 Silver 2.0 checkpoint')
else:
    print('EfficientNetB0 checkpoint not found — using ImageNet weights')
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE,IMG_SIZE,3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)
    eff_full = tf.keras.Model(base.input, out)

# Extract 256-dim penultimate embedding
eff_embed = tf.keras.Model(inputs=eff_full.input, outputs=eff_full.layers[-3].output)
eff_embed.trainable = False
EFF_EMBED_DIM = 256
print(f'EfficientNetB0 embedding dim: {EFF_EMBED_DIM}')

In [ ]:
class SwinClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = timm.create_model('swinv2_tiny_window8_256', pretrained=True, num_classes=0)
        fd = self.backbone.num_features  # 768
        self.head = nn.Sequential(
            nn.Linear(fd,512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512,256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes))
    def forward(self, x): return self.head(self.backbone(x))
    def embed(self, x):
        f = self.backbone(x)
        x = self.head[0](f); x = self.head[1](x); x = self.head[2](x)
        x = self.head[3](x); x = self.head[4](x); x = self.head[5](x)
        return x  # 256-dim

swin_model = SwinClassifier(N_CLASSES).to(DEVICE)
SWIN_CKPT  = f'{CKPT_DIR}/swinv2_silver2_best.pt'
if os.path.exists(SWIN_CKPT):
    swin_model.load_state_dict(torch.load(SWIN_CKPT, map_location=DEVICE))
    print('Loaded SwinV2 Silver 2.0 checkpoint')
else:
    print('SwinV2 checkpoint not found — using ImageNet weights')
for p in swin_model.parameters(): p.requires_grad = False   # frozen
swin_model.eval()
print(f'SwinV2 embedding dim: 256')

In [ ]:
class BiomedCLIPClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        clip_m, _, _ = open_clip.create_model_and_transforms(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
        self.encoder = clip_m.visual
        self.head = nn.Sequential(
            nn.Linear(512,256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes))
    def forward(self, x): return self.head(self.encoder(x))
    def embed(self, x):
        f = self.encoder(x)
        x = self.head[0](f); x = self.head[1](x); x = self.head[2](x)
        return x  # 256-dim

clip_model = BiomedCLIPClassifier(N_CLASSES).to(DEVICE)
CLIP_CKPT  = f'{CKPT_DIR}/biomedclip_silver2_best.pt'
if os.path.exists(CLIP_CKPT):
    clip_model.load_state_dict(torch.load(CLIP_CKPT, map_location=DEVICE))
    print('Loaded BiomedCLIP Silver 2.0 checkpoint')
else:
    print('BiomedCLIP checkpoint not found — using pre-trained weights')
for p in clip_model.parameters(): p.requires_grad = False   # frozen
clip_model.eval()
print(f'BiomedCLIP embedding dim: 256')

## 8. Clinical Image Stream (New in Gold)

The dermoscopic stream re-uses Silver 2.0 models (frozen).
The **clinical stream** is a brand-new EfficientNetB0 trained on MILK10k's clinical photos.

Clinical images have different characteristics from dermoscopic images: no polarized light,
variable magnification, natural lighting, potential occlusion. A separate backbone
lets the model learn these domain-specific features independently.

**Phase 1:** Clinical backbone frozen, only the projector head trained.
**Phase 2:** Clinical backbone unfrozen, full end-to-end fine-tuning.

> **TODO for Kishore:** If MILK10k clinical images aren't available or have quality issues,
> comment out `clin_stream` from the fusion input and reduce `TOTAL_EMBED_DIM` by 256.
> The model will still run with dermoscopic + metadata fusion only.


In [ ]:
class ClinicalStream(nn.Module):
    """
    Separate EfficientNetB0 for clinical (smartphone/closeup) images.
    Outputs a 256-dim embedding matching the Silver 2.0 stream embedding size.
    """
    def __init__(self, embed_dim=256, freeze_backbone=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features   # 1280
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, embed_dim), nn.ReLU(), nn.Dropout(0.3),
        )
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False

    def forward(self, x):
        return self.projector(self.backbone(x))

    def unfreeze(self):
        for p in self.backbone.parameters(): p.requires_grad = True

CLIN_EMBED_DIM = 256
clin_stream = ClinicalStream(embed_dim=CLIN_EMBED_DIM, freeze_backbone=True).to(DEVICE)
trainable   = sum(p.numel() for p in clin_stream.parameters() if p.requires_grad)
total       = sum(p.numel() for p in clin_stream.parameters())
print(f'ClinicalStream | Total params: {total:,} | Phase-1 trainable: {trainable:,}')

## 9. Metadata Encoder

Small MLP: 30-dim → 128 → 64. Uses LayerNorm (not BatchNorm) because most rows
are zero-imputed and BatchNorm statistics would be dominated by the zero entries.


In [ ]:
class MetadataEncoder(nn.Module):
    def __init__(self, in_dim=META_DIM, hidden=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, out_dim), nn.ReLU(), nn.Dropout(0.2),
        )
    def forward(self, x): return self.net(x)

META_EMBED_DIM = 64
meta_encoder   = MetadataEncoder(in_dim=META_DIM, out_dim=META_EMBED_DIM).to(DEVICE)
print(f'MetadataEncoder params: {sum(p.numel() for p in meta_encoder.parameters()):,}')

## 10. Gold Fusion Model

**Late fusion (baseline):** Concatenate all stream embeddings → LayerNorm → MLP → 9 classes.

```
EfficientNetB0 embed (256)  ─┐
SwinV2 embed        (256)  ──┤
BiomedCLIP embed    (256)  ──┼─ concat(1088) → LayerNorm → Linear(512) → ReLU → Dropout
ClinicalStream embed(256)  ──┤                           → Linear(256) → ReLU → Dropout
MetadataEncoder     ( 64)  ──┘                           → Linear(9)
```

A **cross-attention fusion** alternative is provided as a commented-out class below.
Swap it in if late-fusion balanced accuracy plateaus below 0.75.


In [ ]:
TOTAL_EMBED_DIM = EFF_EMBED_DIM + 256 + 256 + CLIN_EMBED_DIM + META_EMBED_DIM  # 1088
print(f'Fusion input dim: {TOTAL_EMBED_DIM}')

class GoldFusionModel(nn.Module):
    def __init__(self, in_dim=TOTAL_EMBED_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),   nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )
    def forward(self, eff, swin, clip_, clin, meta):
        x = torch.cat([eff, swin, clip_, clin, meta], dim=-1)
        return self.classifier(self.norm(x))

gold_model = GoldFusionModel().to(DEVICE)
print(f'GoldFusionModel params: {sum(p.numel() for p in gold_model.parameters()):,}')

# ── Cross-Attention Fusion (stretch goal — swap in if late-fusion plateaus) ────
# class CrossAttentionFusion(nn.Module):
#     """
#     Metadata query attends over image stream key/values.
#     query  = meta_emb projected to d_model
#     key    = stack([eff, swin, clip_, clin]) → (B, 4, d_model)
#     value  = same as key
#     output = attention-weighted context concat meta → classify
#     """
#     def __init__(self, img_dim=256, meta_dim=64, d_model=128, n_heads=4, n_classes=9):
#         super().__init__()
#         self.img_proj  = nn.Linear(img_dim, d_model)
#         self.meta_proj = nn.Linear(meta_dim, d_model)
#         self.attn      = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
#         self.fc        = nn.Sequential(
#             nn.Linear(d_model*2, 256), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(256, n_classes))
#     def forward(self, eff, swin, clip_, clin, meta):
#         imgs   = torch.stack([eff, swin, clip_, clin], dim=1)  # (B,4,256)
#         imgs   = self.img_proj(imgs)                            # (B,4,d_model)
#         q      = self.meta_proj(meta).unsqueeze(1)              # (B,1,d_model)
#         ctx, _ = self.attn(q, imgs, imgs)                      # (B,1,d_model)
#         ctx    = ctx.squeeze(1)
#         return self.fc(torch.cat([ctx, self.meta_proj(meta)], dim=-1))


## 11. Training

**Phase 1 (10 epochs, LR=1e-3):** Silver 2.0 backbones frozen. Clinical backbone frozen.
Train only: clinical projector, metadata encoder, fusion MLP.

**Phase 2 (15 epochs, LR=1e-4):** Unfreeze clinical backbone. Silver 2.0 still frozen.

**Phase 3 (optional, 10 epochs, LR=1e-5):** Unfreeze SwinV2 if Phase 2 BalAcc > 0.73.

> **Note for Kishore on the TF/PyTorch boundary:**
> `eff_embed_batch()` converts PyTorch tensors → NumPy → TF → NumPy → PyTorch.
> It's slow (~30% overhead). To remove this, export EfficientNetB0 to ONNX and load it
> in PyTorch with `onnxruntime`. Optional optimisation, not required for a working result.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

cw_arr = compute_class_weight('balanced', classes=np.arange(N_CLASSES),
                               y=train_meta['label_idx'].values)
for cls, boost in {'mel':1.5,'bcc':1.3,'scc':1.3}.items():
    cw_arr[LABEL2IDX[cls]] *= boost

print('Class weights:', dict(zip(LABEL_NAMES, cw_arr.round(3))))

class FocalLossPT(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return (((1 - torch.exp(-ce))**self.gamma) * ce).mean()

cw_tensor = torch.tensor(cw_arr, dtype=torch.float32).to(DEVICE)
criterion  = FocalLossPT(gamma=2.0, weight=cw_tensor)

# ── EfficientNetB0 batch helper (TF model called from PyTorch training loop) ──
def eff_embed_batch(img_derm_batch):
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
    imgs = ((img_derm_batch.cpu() * std + mean) * 255).clamp(0,255)
    imgs = imgs.permute(0,2,3,1).numpy().astype(np.float32)
    embs = eff_embed(tf.constant(imgs), training=False)
    return torch.tensor(embs.numpy()).to(DEVICE)

@torch.no_grad()
def get_embeddings(img_derm, img_clin, meta):
    img_derm = img_derm.to(DEVICE); img_clin = img_clin.to(DEVICE); meta = meta.to(DEVICE)
    eff_emb  = eff_embed_batch(img_derm)
    swin_emb = swin_model.embed(img_derm)
    clip_emb = clip_model.embed(img_derm)
    clin_emb = clin_stream(img_clin)
    meta_emb = meta_encoder(meta)
    return eff_emb, swin_emb, clip_emb, clin_emb, meta_emb

In [ ]:
def train_epoch(loader, optimizer, trainable_modules):
    for m in trainable_modules: m.train()
    swin_model.eval(); clip_model.eval()
    total_loss = 0; correct = 0; n = 0
    for img_d, img_c, meta, labels in loader:
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        # Note: get_embeddings uses @torch.no_grad for frozen models;
        # clin_stream and meta_encoder gradients DO flow through here.
        eff_emb  = eff_embed_batch(img_d)         # no grad (TF)
        swin_emb = swin_model.embed(img_d.to(DEVICE))  # no grad (frozen)
        clip_emb = clip_model.embed(img_d.to(DEVICE))  # no grad (frozen)
        clin_emb = clin_stream(img_c.to(DEVICE))        # grad flows
        meta_emb = meta_encoder(meta.to(DEVICE))        # grad flows
        logits   = gold_model(eff_emb, swin_emb, clip_emb, clin_emb, meta_emb)
        loss     = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(
            [p for m in trainable_modules for p in m.parameters()], 1.0)
        optimizer.step()
        total_loss += loss.item()*len(labels); correct += (logits.argmax(1)==labels).sum().item(); n += len(labels)
    return total_loss/n, correct/n

@torch.no_grad()
def evaluate(loader):
    gold_model.eval(); clin_stream.eval(); meta_encoder.eval()
    preds, trues, probs = [], [], []
    for img_d, img_c, meta, labels in loader:
        eff_emb  = eff_embed_batch(img_d)
        swin_emb = swin_model.embed(img_d.to(DEVICE))
        clip_emb = clip_model.embed(img_d.to(DEVICE))
        clin_emb = clin_stream(img_c.to(DEVICE))
        meta_emb = meta_encoder(meta.to(DEVICE))
        logits   = gold_model(eff_emb, swin_emb, clip_emb, clin_emb, meta_emb)
        probs.append(torch.softmax(logits,-1).cpu().numpy())
        preds.append(logits.argmax(1).cpu().numpy())
        trues.append(labels.numpy())
    y_pred = np.concatenate(preds); y_true = np.concatenate(trues); y_prob = np.concatenate(probs)
    return balanced_accuracy_score(y_true, y_pred), y_pred, y_true, y_prob

In [ ]:
# ── Phase 1: Fusion layer + clinical head + metadata encoder ──────────────────
print('=== Phase 1: Frozen backbones — training fusion layer only ===')
p1_modules = [gold_model, clin_stream, meta_encoder]
p1_params  = [p for m in p1_modules for p in m.parameters() if p.requires_grad]
opt1   = torch.optim.AdamW(p1_params, lr=1e-3, weight_decay=1e-4)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=10)
best_val = 0.0

for epoch in range(1, 11):
    tr_loss, tr_acc = train_epoch(train_loader, opt1, p1_modules)
    val_bal, *_ = evaluate(val_loader)
    sched1.step()
    print(f'Ep {epoch:2d}/10 | loss={tr_loss:.4f} acc={tr_acc:.4f} val_bal={val_bal:.4f}')
    if val_bal > best_val:
        best_val = val_bal
        torch.save(gold_model.state_dict(),   f'{GOLD_DIR}/gold_fusion_best.pt')
        torch.save(clin_stream.state_dict(),  f'{GOLD_DIR}/gold_clin_best.pt')
        torch.save(meta_encoder.state_dict(), f'{GOLD_DIR}/gold_meta_best.pt')
        print(f'  >> Checkpoint saved (val BalAcc={best_val:.4f})')

print(f'Phase 1 done. Best val BalAcc: {best_val:.4f}')

In [ ]:
# ── Phase 2: Unfreeze clinical backbone ─────────────────────────────────────────
print('=== Phase 2: Unfreeze clinical backbone ===')
clin_stream.unfreeze()
p2_modules = [gold_model, clin_stream, meta_encoder]
p2_params  = [p for m in p2_modules for p in m.parameters() if p.requires_grad]
opt2   = torch.optim.AdamW(p2_params, lr=1e-4, weight_decay=1e-4)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15)
patience = 0

for epoch in range(1, 16):
    tr_loss, tr_acc = train_epoch(train_loader, opt2, p2_modules)
    val_bal, *_ = evaluate(val_loader)
    sched2.step()
    print(f'Ep {epoch:2d}/15 | loss={tr_loss:.4f} acc={tr_acc:.4f} val_bal={val_bal:.4f}')
    if val_bal > best_val:
        best_val = val_bal; patience = 0
        torch.save(gold_model.state_dict(),   f'{GOLD_DIR}/gold_fusion_best.pt')
        torch.save(clin_stream.state_dict(),  f'{GOLD_DIR}/gold_clin_best.pt')
        torch.save(meta_encoder.state_dict(), f'{GOLD_DIR}/gold_meta_best.pt')
        print(f'  >> Checkpoint saved (val BalAcc={best_val:.4f})')
    else:
        patience += 1
        if patience >= 5: print('Early stopping.'); break

print(f'Phase 2 done. Best val BalAcc: {best_val:.4f}')

## 12. Evaluation — Test Set

In [ ]:
gold_model.load_state_dict(torch.load(f'{GOLD_DIR}/gold_fusion_best.pt', map_location=DEVICE))
clin_stream.load_state_dict(torch.load(f'{GOLD_DIR}/gold_clin_best.pt', map_location=DEVICE))
meta_encoder.load_state_dict(torch.load(f'{GOLD_DIR}/gold_meta_best.pt', map_location=DEVICE))
gold_model.eval(); clin_stream.eval(); meta_encoder.eval()

test_bal, y_pred, y_true, y_prob = evaluate(test_loader)
print(f'=== Gold Model — Test Set ===')
print(f'Balanced Accuracy: {test_bal:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

auroc_scores = {}
for i, name in enumerate(LABEL_NAMES):
    try: auroc_scores[name] = roc_auc_score((y_true==i).astype(int), y_prob[:,i])
    except: auroc_scores[name] = float('nan')
macro_auroc = np.nanmean(list(auroc_scores.values()))
print(f'\nMacro AUROC: {macro_auroc:.4f}')
for n, s in auroc_scores.items(): print(f'  {n:<8}: {s:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))
cm = confusion_matrix(y_true, y_pred, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES, cmap='Blues', ax=axes[0])
axes[0].set_title('Gold — Normalised Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

recalls = cm.diagonal()
colours = ['#d62728' if n in HIGH_RISK else '#ff7f0e' if n in MOD_RISK else '#1f77b4'
           for n in LABEL_NAMES]
axes[1].bar(LABEL_NAMES, recalls, color=colours)
axes[1].axhline(0.80, color='red',    ls='--', label='High-risk target (0.80)')
axes[1].axhline(0.75, color='orange', ls='--', label='BalAcc target (0.75)')
axes[1].set_title('Gold — Per-Class Recall'); axes[1].set_ylim(0,1); axes[1].legend()
plt.tight_layout(); plt.show()

## 13. Fairness Audit — Fitzpatrick17k & DDI

Same framework as Silver 2.0. Key question: does metadata fusion help?
- For PAD-UFES-20 images (explicit Fitzpatrick labels): likely yes
- For ISIC 2019 images (zero-imputed metadata): marginal change expected


In [ ]:
fitz_path = kagglehub.dataset_download('nazmusresan/fitzpatrick17k')
ddi_path  = kagglehub.dataset_download('souvikda/ddidiversedermatology-multimodal-dataset')

FITZ_MAP = {
    'melanoma':'mel','lentigo maligna':'mel','lentigo maligna melanoma':'mel',
    'melanocytic nevi':'nv','nevus':'nv','blue nevus':'nv',
    'basal cell carcinoma':'bcc',
    'squamous cell carcinoma':'scc','squamous cell carcinoma in situ':'akiec',
    'actinic keratosis':'akiec','bowen s disease':'akiec',
    'seborrheic keratosis':'bkl','benign keratosis':'bkl',
    'dermatofibroma':'df','vascular lesion':'vasc','angioma':'vasc',
    'tinea':'unk','psoriasis':'unk','eczema':'unk',
}

fitz_csv = None
for fname in ['fitzpatrick17k.csv','metadata.csv']:
    p = Path(fitz_path)/fname
    if p.exists(): fitz_csv = pd.read_csv(p); break

fitz_img_dir = Path(fitz_path)
for d in ['images','data','']:
    p = Path(fitz_path)/d
    if p.is_dir() and any(p.glob('*.jpg')): fitz_img_dir = p; break

fitz_csv['label_mapped'] = fitz_csv['label'].str.lower().map(FITZ_MAP).fillna('unk')
fitz_csv['label_idx']    = fitz_csv['label_mapped'].map(LABEL2IDX)
fitz_csv = fitz_csv.dropna(subset=['label_idx']); fitz_csv['label_idx'] = fitz_csv['label_idx'].astype(int)
fitz_csv['path_derm'] = fitz_csv['md5hash'].apply(lambda x: str(fitz_img_dir/f'{x}.jpg'))
fitz_csv['path_clin'] = fitz_csv['path_derm']
fitz_csv['fitzpatrick'] = pd.to_numeric(fitz_csv.get('fitzpatrick', np.nan), errors='coerce')

valid = fitz_csv[fitz_csv['path_derm'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
print(f'Fitzpatrick17k valid images: {len(valid):,}')

fitz_feats  = encode_metadata(valid)
fitz_ds     = GoldDataset(valid.rename(columns={'path_derm':'path_derm','path_clin':'path_clin'}),
                           fitz_feats, val_transform)
fitz_loader = DataLoader(fitz_ds, BATCH_SIZE, shuffle=False, num_workers=2)

_, fitz_preds, fitz_true, _ = evaluate(fitz_loader)
valid['pred'] = fitz_preds

print('\n=== Gold Fairness Audit — Fitzpatrick17k ===')
print(f'  {"Fitz":>4}  {"N":>6}  {"Accuracy":>10}  {"Mel Recall":>10}  {"BCC Recall":>10}')
print('  ' + '-'*52)
mel_idx = LABEL2IDX['mel']; bcc_idx = LABEL2IDX['bcc']
acc_gaps = []; mel_gaps = []
for ft in sorted(valid['fitzpatrick'].dropna().unique()):
    sub = valid[valid['fitzpatrick']==ft]
    if len(sub) < 10: continue
    acc = (sub['pred']==sub['label_idx']).mean()
    mel_r = (sub.loc[sub['label_idx']==mel_idx,'pred']==mel_idx).mean() if (sub['label_idx']==mel_idx).sum()>0 else float('nan')
    bcc_r = (sub.loc[sub['label_idx']==bcc_idx,'pred']==bcc_idx).mean() if (sub['label_idx']==bcc_idx).sum()>0 else float('nan')
    acc_gaps.append(acc); mel_gaps.append(mel_r)
    print(f'  {int(ft):>4}  {len(sub):>6}  {acc:>10.4f}  {mel_r:>10.4f}  {bcc_r:>10.4f}')
ag = max(acc_gaps)-min(acc_gaps); mg = np.nanmax(mel_gaps)-np.nanmin(mel_gaps)
print('  ' + '-'*52)
print(f'  Max accuracy gap  : {ag:.4f}  (target < 0.05)')
print(f'  Max mel-recall gap: {mg:.4f}  (target < 0.10)')
print(f'  Equity (acc)      : {"PASS" if ag<0.05 else "FAIL"}')
print(f'  Equity (mel)      : {"PASS" if mg<0.10 else "FAIL"}')

## 14. Final Comparison — Bronze → Silver → Silver 2.0 → Gold

In [ ]:
# Silver 2.0 actual results (April 2026 run)
results = {
    'Bronze EfficientNetB0 (HAM10000)':      {'Bal':'--','AUROC':'--','Mel':'--','BCC':'--','SCC':'--','FitzGap':'--'},
    'Silver EfficientNetB0':                  {'Bal':0.3717,'AUROC':0.8686,'Mel':0.5311,'BCC':0.0434,'SCC':0.7094,'FitzGap':0.1636},
    'Silver SwinV2':                          {'Bal':0.7132,'AUROC':0.9574,'Mel':0.7613,'BCC':0.7475,'SCC':0.6000,'FitzGap':0.3582},
    'Silver BiomedCLIP':                      {'Bal':0.6024,'AUROC':0.9300,'Mel':0.7966,'BCC':0.6534,'SCC':0.5660,'FitzGap':0.1928},
    'Silver 2.0 Weighted Ensemble':           {'Bal':0.7208,'AUROC':0.9571,'Mel':0.7881,'BCC':0.7330,'SCC':0.6679,'FitzGap':0.3498},
    'Gold Multimodal Fusion':                 {'Bal':None,'AUROC':None,'Mel':None,'BCC':None,'SCC':None,'FitzGap':None},
}

# Fill in Gold results
gold_recalls = recall_score(y_true, y_pred, average=None, labels=list(range(N_CLASSES)), zero_division=0)
results['Gold Multimodal Fusion'] = {
    'Bal': test_bal, 'AUROC': macro_auroc,
    'Mel': gold_recalls[LABEL2IDX['mel']], 'BCC': gold_recalls[LABEL2IDX['bcc']],
    'SCC': gold_recalls[LABEL2IDX['scc']], 'FitzGap': ag,
}

def fmt(v): return f'{v:.4f}' if isinstance(v, float) else str(v)
print('='*105)
print('  DermaVision AI — Full Model Comparison')
print('='*105)
print(f'  {"Model":<40} {"BalAcc":>7} {"AUROC":>7} {"Mel":>7} {"BCC":>7} {"SCC":>7} {"FitzGap":>9}')
print('  '+'-'*105)
for name, r in results.items():
    print(f'  {name:<40} {fmt(r["Bal"]):>7} {fmt(r["AUROC"]):>7} '
          f'{fmt(r["Mel"]):>7} {fmt(r["BCC"]):>7} {fmt(r["SCC"]):>7} {fmt(r["FitzGap"]):>9}')
print('='*105)
print()
print('  Targets: Mel>0.80 | BCC>0.75 | SCC>0.75 | BalAcc>0.75 | AUROC>0.90 | FitzGap<0.05')

## 15. Streamlit App Scaffold

Run this cell to write `gold_app.py`, then launch with streamlit.

> **TODO for Kishore:** In the `run_inference()` function, load the Gold checkpoints
> and connect `get_embeddings()` to the uploaded image. The UI is complete —
> you just need to replace the placeholder predictions with real model output.


In [ ]:
app_code = '''
import streamlit as st
import torch, numpy as np
from PIL import Image
import os, json

st.set_page_config(page_title="DermaVision AI", layout="wide", page_icon="🔬")
st.title("🔬 DermaVision AI — Gold Model")
st.caption("Multimodal skin lesion classifier | Research use only")
st.error("NOT a clinical diagnostic tool. Never use for medical decision-making.")

LABEL_NAMES = ["mel","nv","bcc","scc","akiec","bkl","df","vasc","unk"]
RISK = {"mel":"HIGH","bcc":"HIGH","scc":"HIGH","akiec":"MODERATE",
        "nv":"Low","bkl":"Low","df":"Low","vasc":"Low","unk":"—"}
RISK_COLOUR = {"HIGH":"🔴","MODERATE":"🟡","Low":"🟢","—":"⚪"}
RISK_DESC = {
    "mel":"Melanoma — urgent referral required",
    "bcc":"Basal cell carcinoma — needs excision",
    "scc":"Squamous cell carcinoma — needs excision",
    "akiec":"Actinic keratosis — monitor, possible treatment",
    "nv":"Melanocytic nevus — benign, routine check",
    "bkl":"Benign keratosis — benign, cosmetic concern only",
    "df":"Dermatofibroma — benign",
    "vasc":"Vascular lesion — benign",
    "unk":"Unclassified — manual review recommended",
}

with st.sidebar:
    st.header("Patient Metadata")
    st.caption("Providing metadata improves accuracy, especially for high-risk classes.")
    age      = st.slider("Age", 0, 100, 45)
    sex      = st.selectbox("Sex", ["unknown","male","female"])
    fitz     = st.selectbox("Fitzpatrick Skin Type",
                            ["unknown","I","II","III","IV","V","VI"])
    location = st.selectbox("Lesion Location",
                            ["unknown","back","face","chest","lower extremity",
                             "upper extremity","trunk","abdomen","foot","hand"])
    diameter = st.slider("Estimated Lesion Diameter (mm)", 0, 50, 10)

uploaded = st.file_uploader("Upload a dermoscopic or clinical skin lesion image",
                             type=["jpg","jpeg","png"])

if uploaded:
    img = Image.open(uploaded).convert("RGB")
    col1, col2 = st.columns([1,2])
    col1.image(img, caption="Uploaded image", use_column_width=True)

    with st.spinner("Running Gold model inference..."):
        # TODO: replace with real model inference
        # 1. Load models with @st.cache_resource
        # 2. encode_metadata() the sidebar inputs
        # 3. Preprocess img → tensor
        # 4. get_embeddings(img_derm, img_clin, meta) → gold_model(...)
        # 5. probs = softmax(logits)
        probs = np.random.dirichlet(np.ones(len(LABEL_NAMES)))  # placeholder
        st.warning("Placeholder output — connect real model in run_inference()")

    top3 = np.argsort(probs)[::-1][:3]
    with col2:
        st.subheader("Top Predictions")
        for idx in top3:
            name = LABEL_NAMES[idx]
            st.markdown(f"{RISK_COLOUR[RISK[name]]} **{name.upper()}** — {probs[idx]*100:.1f}%  _{RISK_DESC[name]}_")
            st.progress(float(probs[idx]))

    st.subheader("Full Distribution")
    import pandas as pd
    df = pd.DataFrame({"Class":LABEL_NAMES,"Prob":probs}).sort_values("Prob",ascending=True)
    st.bar_chart(df.set_index("Class"))

    st.info("Grad-CAM overlay: TODO — display pytorch-grad-cam output on the uploaded image")

st.divider()
st.caption("DermaVision AI | MSBC 5190 Group 7 | Spring 2026")
\'\'\'

with open('/content/gold_app.py', 'w') as f: f.write(app_code)
print('gold_app.py written.')
print()
print('To launch:')
print('  !pip install -q streamlit pyngrok')
print('  !streamlit run /content/gold_app.py &')
print('  # Use ngrok or Colab port forwarding to expose the URL')


## 16. Notes for Kishore

### What is fully written (just run the cells)
| Section | Status |
|---------|--------|
| Data pipeline (ISIC 2019, PAD-UFES-20, MILK10k paired) | Ready — may need path tweaks for MILK10k |
| Metadata encoding (30-dim, zero-imputation + missingness flags) | Ready |
| GoldDataset — paired image + metadata DataLoader | Ready |
| Silver 2.0 backbone loading (EfficientNetB0, SwinV2, BiomedCLIP) | Ready |
| ClinicalStream — EfficientNetB0 for clinical images | Ready |
| MetadataEncoder — 30→64 MLP with LayerNorm | Ready |
| GoldFusionModel — late-fusion 1088→512→256→9 | Ready |
| Phase 1 & 2 training loops with early stopping | Ready |
| Evaluation — BalAcc, AUROC, per-class recall, confusion matrix | Ready |
| Fairness audit — Fitzpatrick17k + DDI | Ready |
| Final comparison table (Silver 2.0 numbers pre-filled) | Ready |
| Streamlit app scaffold | Ready |

### What needs your work (TODOs)
| Task | Location | Notes |
|------|----------|-------|
| MILK10k path debugging | Section 3, MILK10k cells | Print `milk_root.iterdir()` and inspect; adjust `derm_dir`/`clin_dir` |
| Streamlit model inference | `gold_app.py`, `run_inference()` | Load checkpoints, call `get_embeddings()`, display Grad-CAM |
| Cross-attention fusion (stretch) | Section 10, commented-out class | Swap in if late-fusion BalAcc < 0.73 after Phase 2 |
| Phase 3 SwinV2 fine-tuning (stretch) | After Phase 2 | Unfreeze SwinV2 at LR=1e-5 for 10 epochs |

### Practical tips
- If Colab RAM runs out: reduce `BATCH_SIZE` from 32 to 16
- Run Silver 2.0 notebook first to generate the checkpoint files this notebook loads
- The `eff_embed_batch()` TF/PyTorch bridge is slow — acceptable for now, ONNX export removes the overhead
- Save checkpoints to Drive often — Colab disconnects without warning
- Monitor **val balanced accuracy** (not overall accuracy) — same priority as Silver 2.0
- If MILK10k clinical images are unavailable: set `CLIN_EMBED_DIM = 0`, remove `clin_emb` from `GoldFusionModel.forward()`, and set `TOTAL_EMBED_DIM = 832`. The model will still run on image+metadata fusion.

### Expected Gold results (rough targets)
| Metric | Silver 2.0 | Gold (expected) |
|--------|-----------|-----------------|
| BalAcc | 0.721 | 0.730–0.760 |
| Mel Recall | 0.788 | 0.790–0.820 |
| BCC Recall | 0.733 | 0.740–0.780 |
| FitzGap | 0.350 | 0.300–0.340 (metadata helps PAD-UFES rows) |

The Fitzpatrick gap will likely remain large because ISIC 2019 dominates the training set
and provides no metadata. The metadata stream primarily helps for PAD-UFES-20 rows (~6% of training data).
